# Bus backups and efficiency in Madison, WI
### Alton Hipps

## Data ingest
#### Import necessary modules

In [1]:
import geopandas as gpd
import pandas as pd
import numpy as np
from osgeo import gdal
import os
from datetime import datetime,timedelta

#!pip install keplerGL
import keplergl

#### Access Data Files and Combine by Bus Route

In [2]:
# Timestamp function
def ts():
    return datetime.now().strftime("%X.%f")

# Short Conditional Print function
def scp(message,boolVal=True):
    if boolVal==True:
        print(f'{ts()}\t{message}')
    else:
        return

#### Define functions to read in data and skip junk lines

In [3]:
def nextReadableLine(fileObj):
    try:
        csvList=fileObj.readline()
        return csvList
    except:
        return (nextReadableLine(fileObj))

def fileToListOfLists(csvPath):
    fileList=[]
    f=open(csvPath)
    while True:
        line=nextReadableLine(f)
        if not line:
            break
        else:
            lineList=line.split(',')
            fileList.append(lineList)
    f.close()
    return fileList

#### Define class to handle a bus route and create a dataframe for it

In [4]:
# create class to contain data for each bus destination
class BusRoute:
    count=1
    def __init__(self):
        self.id=BusRoute.count
        #self.df=[]
        BusRoute.count+=1

    # Method to read a csv to a dataframe
    def fromCSV(self,csvPath,headDictionary,mode='new'):
        eCount=1
        longList=fileToListOfLists(csvPath)
        # Package as dataframe
        skip=True
        for listItem in longList:
            skip=False
            if len(listItem)!=10:
                skip=True
                continue
            try:
                headDict=dict(headDictionary)
                headDict['TimeStamp'].append(datetime(
                    int(listItem[0][:4]), # year
                    int(listItem[0][4:6]), # month
                    int(listItem[0][6:8]), # day
                    int(listItem[0].split(' ')[1]), # hour
                    int(listItem[0].split(' ')[2]), # minute
                    int(listItem[0].split(' ')[3]) # second
                ))
                headDict['Lat'].append(listItem[1])
                headDict['Long'].append(listItem[2])
                headDict['Heading'].append(listItem[3])
                headDict['RTDesignation'].append(listItem[4])
                headDict['Destination'].append(listItem[5])
                headDict['DelayStatus'].append(listItem[6])
                headDict['Speed'].append(listItem[7])
                headDict['Fullness'].append(listItem[8])
                headDict['BusID'].append(listItem[9][:-1])
            except Exception as e:
                if(len(listItem[0]))>20:listItem='Too long'
                #print(f'{eCount}\t{listItem}')
                #print(f'\t{e}')
                eCount+=1
                continue
        if skip==True:
            #print('No data ingested')
            return False
        if len(headDict)<1:
            #print('No data ingested')
            return False
        outDf=pd.DataFrame(headDict)
        if mode=='new':
            #print('New!!')
            self.df=pd.DataFrame(headDict)
            return True
        elif mode=='add':
            oldDf=self.df
            oldDf.append(outDf)
            self.df=oldDf
            #print('Add!!')
            return True
        else:
            #print('DF Only')
            return outDf

#### Define functions to search folder structure for csvs and to create the BusRoute objects from those csvs

In [28]:
# function to find all csvs
def folderSearch(dataFolder):
    folderStructure=[]
    # Identity Route Name folders
    bigList=os.listdir(dataFolder)
    
    for item in bigList:
        routeList=os.listdir(dataFolder+'\\'+item)
        for route in routeList:
            destinationList=os.listdir(dataFolder+'\\'+item+'\\'+route)
            if len(destinationList)!=0:
                for dest in destinationList:
                    folderStructure.append(dataFolder+'\\'+item+'\\'+route+'\\'+dest)
    return folderStructure

# function to identify last x days of csvs
def timegateFiles(folderStructure,numOfDays=14):
    today=datetime.today()
    month=today.month
    if month<10:
        month='0'+str(month)
    todayFile=f'{str(today.year)[2:4]}_{month}_{today.day}.csv'
    legalDates=[]
    count=0
    dayNum=today.day
    while count<numOfDays:
        
        if dayNum <10:
            dayForStr = '0'+str(dayNum)
        else:
            dayForStr = dayNum
        
        dayStr=f'{str(today.year)[2:4]}_{month}_{dayForStr}.csv'
        legalDates.append(dayStr)
        dayNum=dayNum-1
        
        if dayNum < 1:
            if month in ['02','04','06','08','11','01']:
                dayNum=31
            if month == '03':
                dayNum=28
            else:
                dayNum=30
            month=str(int(month)-1)
            if int(month) < 1:
                month='12'
                year=year-1
            if int(month) < 10:
                month='0'+month
        count+=1
    #print(legalDates)
    
    # loop through files to gate them
    fileDict={}
    lastFile=''
    for filePath in folderStructure:
        route=filePath.split('\\')[1]
        date=filePath.split('\\')[3]
        
        if date not in legalDates:
            continue
        
        if route in fileDict.keys():
            tempList=fileDict[route]
            tempList.append(filePath)
            fileDict[route]=tempList
        else:
            fileDict[route]=[filePath]
    return fileDict
    

# function to create BusRoute objs for each route
def createBusRoutes(dataFolder,headers):
    busRoutes={}
    counter=0
    allPaths=dataFolder
    
    for path in allPaths:
        spliter=path.split('\\')
        routeName=spliter[1]
        destination=spliter[2]
        fileName=spliter[3]

        # add each bus route
        if routeName not in busRoutes.keys():
            busRoutes[routeName]={}

        # add each destination
        routeDict=busRoutes[routeName]

        busObj=BusRoute()
        message=busObj.fromCSV(path,headers)
        if message==True:
            busRoutes[routeName][destination]=busObj.df
        
        ''' for testing
        print(counter)
        counter+=1
        if counter>10:
            break
        '''     
    return busRoutes

#### Define function to create config dictionary for Kepler GL

In [60]:
def createConfig(rtDes,dateRange):
    outConfig={
        'version': 'v1',
        "config":{
            "visState":{
                "filters":[{
                    "dataId":[f"Rt{rtDes}"],
                    "id":f"Rt{rtDes}_timestamp",
                    "name":["TimeStamp"],
                    "type":"timeRange", 
                    "value":[dateRange[0],dateRange[1]],
                    "plotType":{
                        "interval":"6-hour",
                        "defaultTimeFormat":"L  H A",
                        "type":"histogram",
                        "aggregation":"sum"
                        },
                    "animationWindow":"free",
                    "yAxis":'null',
                    "view":"enlarged",
                    "speed":0.05,
                    "enabled":'true'
                    }
                ],
                "layers":[{
                    "id":f"-Rt{rtDes}_id",
                    "type":"geojson",
                    "config":{
                        "dataId":f"Rt{rtDes}",
                        "columnMode":"geojson",
                        "label":f"Bus Fullness - Route {rtDes}",
                        "color":[34,63,154],
                        "highlightColor":[252,242,26,255],
                        "columns":{"geojson":"geometry"},
                        "isVisible":'true',
                        "visConfig":{
                            "opacity":0.8,
                            "strokeOpacity":0.8,
                            "thickness":0.5,
                            "strokeColor":'null',
                            "colorRange":{
                                "colors":["#FAE300","#CF1750","#223F9A"],
                                "name":"UberPool",
                                "type":"diverging",
                                "category":"Uber",
                                "reversed":True,
                                "colorMap":[["EMPTY","#FAE300"],["FULL","#CF1750"],["HALF_EMPTY","#223F9A"]]
                                },
                            "strokeColorRange":{
                                "name":"Global Warming",
                                "type":"sequential",
                                "category":"Uber",
                                "colors":[
                                    "#4C0035",
                                    "#880030",
                                    "#B72F15",
                                    "#D6610A",
                                    "#EF9100",
                                    "#FFC300"
                                    ]
                                },
                            "radius":10,
                            "sizeRange":[0,10],
                            "radiusRange":[0,50],
                            "heightRange":[0,500],
                            "elevationScale":5,
                            "stroked":False,
                            "filled":True,
                            "enable3d":False,
                            "wireframe":False,
                            "fixedHeight":False
                            },
                        "hidden":False,
                        "textLabel":[{
                            "field":'Fullness',
                            "color":[255,255,255],
                            "size":18,
                            "offset":[0,0],
                            "anchor":"start",
                            "alignment":"center",
                            "outlineWidth":0,
                            "outlineColor":[255,0,0,255],
                            "background":'false',
                            "backgroundColor":[0,0,200,255]
                            }]
                    },
                    "visualChannels":{
                        "colorField":{
                            "name":"Fullness","type":"string"
                        },
                        "colorScale":"customOrdinal",
                        "strokeColorField":'null',
                        "strokeColorScale":"quantile",
                        "sizeField":'null',
                        "sizeScale":"linear",
                        "heightField":'null',
                        "heightScale":"linear",
                        "radiusField":'null',
                        "radiusScale":"linear"
                        }
                }],
                "effects":[],
                "interactionConfig":{
                    "tooltip":{
                        "fieldsToShow":{
                            "-RtA_id":[{
                                    "name":"DelayStatus",
                                    "format":'null'
                                },{
                                    "name":"Destination",
                                    "format":'null'
                                },{
                                    "name":"Fullness",
                                    "format":'null'
                                },{
                                    "name":"Speed",
                                    "format":'null'
                                }]
                        },
                        "compareMode":False,
                        "compareType":"absolute",
                        "enabled":True
                    },
                    "brush":{
                        "size":0.5,
                        "enabled":False
                    },
                    "geocoder":{
                        "enabled":False
                    },
                    "coordinate":{
                        "enabled":False
                    }
                },
                "layerBlending":"additive",
                "overlayBlending":"screen",
                "splitMaps":[],
                "animationConfig":{
                    "currentTime":'null',
                    "speed":1
                },
                "editor":{
                    "features":[],
                    "visible":False
                }
            },
            "mapState":{
                "bearing":0,
                "dragRotate":False,
                "latitude":43.07794042346403,
                "longitude":-89.37636850809596,
                "pitch":20,
                "zoom":11,
                "isSplit":False,
                "isViewportSynced":True,
                "isZoomLocked":False,
                "splitMapViewports":[]
            },'''
            "mapStyle":{
                "styleType":"dark-matter",
                "topLayerGroups":{},
                "visibleLayerGroups":{
                    "label":True,
                    "road":True,
                    "border":False,
                    "building":True,
                    "water":True,
                    "land":True,
                    "3d building":False
                },
                "threeDBuildingColor":[15.035172933000911,15.035172933000911,15.035172933000911],
                "backgroundColor":[0,0,0],
                "mapStyles":{}
            },'''
            "uiState":{
                "mapControls":{
                    "mapLegend":{
                        "active":True
                    }
                }
            }
        }
    }
    return outConfig

## Map Creation
#### Define constants to use for this use case

In [29]:
# Constants
headers={
    'TimeStamp':[],
    'Lat':[],
    'Long':[],
    'Heading':[],
    'RTDesignation':[],
    'Destination':[],
    'DelayStatus':[],
    'Speed':[],
    'Fullness':[],
    'BusID':[]
}

folderPath=r"combined"

#### Create single csv for each map to create
##### Below cell takes a long time to run (up to and over an hour)

In [36]:
allPaths=folderSearch(folderPath)
organizedFiles=timegateFiles(allPaths)
routeList=organizedFiles.keys()

for route in routeList:
    rt_buses=createBusRoutes(organizedFiles[route],headers)
    dictOfDfs=rt_buses
    
    first=True
    for key in dictOfDfs.keys():
        for key2 in dictOfDfs[key].keys():
            #print(key,key2)
            if first==True:
                comboDF=dictOfDfs[key][key2].copy(deep=True)
                #print(comboDF.size)
                first=False
            else:
                comboDF=pd.concat([comboDF,dictOfDfs[key][key2]])
    comboDF.reset_index()
    comboDF.to_csv(f'mapCreation\\{route}s.csv')

#### Read files and create maps

In [73]:
for route in routeList:
    print(f'Started Route {route}')
    fullPD=pd.read_csv(f'mapCreation\\{route}s.csv', low_memory=False)
    fullPD=fullPD.drop(columns='Unnamed: 0')
    fullPD.infer_objects()
    filterDf=fullPD[fullPD['RTDesignation']==route]
    filterDf.reset_index()
    
    # Create geodataframe from lat, long data
    gdf = gpd.GeoDataFrame(filterDf,  # convert the dataframe to geodataframe
                            geometry=gpd.points_from_xy(x=filterDf.Long,
                                                        y=filterDf.Lat))
    gdf=gdf.drop(['Lat','Long'],axis=1)
    gdf=gdf.drop_duplicates()
    
    # Find the largest and smallest date to put bounds on timeslider
    maxDate=gdf['TimeStamp'].max()
    maxDateObj=datetime(year=int(maxDate[0:4]),month=int(maxDate[5:7]),day=int(maxDate[8:10]))
    unixMax = int(str(maxDateObj.timestamp())[:-2]+'00')
    minDate=gdf['TimeStamp'].min()
    minDateObj=datetime(year=int(minDate[0:4]),month=int(minDate[5:7]),day=int(minDate[8:10]))
    unixMin = int(str(minDateObj.timestamp())[:-2]+'00')
    dtRange=(unixMin,unixMax)
    filtered_gdf=gdf.iloc[:, [2,3,4,5,6,1,0,7,8]]
    
    #Additional filtering necessary to confirm small enough final file size
    print(filtered_gdf.shape[0])
    maxRecCount=180000
    if filtered_gdf.shape[0]>maxRecCount:
        filtered_gdf=filtered_gdf.sort_values(by='TimeStamp',ascending=False).reset_index()
        filtered_gdf=filtered_gdf.drop(['index'],axis=1)
        filtered_gdf=filtered_gdf.loc[0:maxRecCount-1]
        try:
            filtered_gdf=filtered_gdf.drop(['level_0'], axis=1)
        except:
            print(Exception)
    
    # Create the map
    mapObj = keplergl.KeplerGl(config=createConfig(route,dtRange), show_docs=False)
    mapObj.add_data(data=filtered_gdf, name=f'Rt{route}')

    mapObj.save_to_html(file_name=f'routes\\{route}.html',read_only=True)
    print(f'Completed Route {route}')

Started Route B
220736
<class 'Exception'>
Map saved to routes\B.html!
Completed Route B
Started Route D
307061
<class 'Exception'>
Map saved to routes\D.html!
Completed Route D
